# Formula.vLLM — reported vs reproduced gaps

Reads `results/<pr>.json` (written by `record.py` → `derive.py`). **gap = PR's solution − baseline**, taken relative to its own baseline so units cancel
(`(baseline − PR's solution) ÷ baseline`; throughput units flipped). **gap ratio = reproduced gap ÷ reported gap** (1 = we see exactly the author's gap).
Only real runs count (`summary.measured > 0`, `ts ≥ 2026-11-07`).

In [ ]:
import json, glob, os
import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath('stats.ipynb'))
rows = []
for f in sorted(glob.glob(os.path.join(HERE, 'results', '[0-9]*.json'))):
    d = json.load(open(f)); s = d.get('summary') or {}
    if not s.get('measured') or d.get('ts', '') < '2026-11-07':
        continue
    h = s.get('headline_derived') or {}; rp, rr = h.get('reproduced', {}), h.get('reported', {})
    rows.append(dict(pr=d['pr'], field=d.get('field'), headline=s.get('headline'), unit=h.get('unit'),
                     reported_baseline=rr.get('baseline'), reported_pr_solution=rr.get('pr_solution'), reported_ratio=rr.get('ratio'), reported_marker=rr.get('marker'),
                     reproduced_baseline=rp.get('baseline'), reproduced_pr_solution=rp.get('pr_solution'), reproduced_ratio=rp.get('ratio'), reproduced_ratio_sd=rp.get('ratio_sd'),
                     ratio_of_ratios=(h.get('reproduced_vs_reported') or {}).get('ratio_of_ratios'), verdict=s.get('verdict'),
                     reliability=(h.get('reliability') or {}).get('flag'), tests=f"{(s.get('tests') or {}).get('base')}/{(s.get('tests') or {}).get('gold')}",
                     title=d.get('title')))
df = pd.DataFrame(rows).sort_values('pr').reset_index(drop=True)
up = df.unit.fillna('').str.contains(r'/s$|tok/s|req/s')
gap = lambda r: np.where(up, r - 1, 1 - 1 / r)          # relative gap from a speedup ratio
df['reported_gap'] = gap(df.reported_ratio.astype(float)); df['reproduced_gap'] = gap(df.reproduced_ratio.astype(float))
df['gap_ratio'] = df.reproduced_gap / df.reported_gap
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 40)
print(f"{len(df)} PRs reproduced here · {int((df.verdict=='reproduced').sum())} match the reported ratio (≥0.8 of it)")
df[['pr','field','headline','unit','reported_baseline','reported_pr_solution','reported_marker','reproduced_baseline','reproduced_pr_solution','reported_gap','reproduced_gap','gap_ratio','verdict','reliability','tests']]

In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE = '#0b8ce8', '#e39b00'   # reproduced / reported (same as the dashboard)
x = np.arange(len(df)); lab = [f"#{p}" for p in df.pr]
fig, ax = plt.subplots(2, 2, figsize=(14, 9), gridspec_kw={'width_ratios': [3, 2]})

# 1. gap ratio per PR (log y, 1 = exactly the author's gap)
a = ax[0, 0]; m = df.gap_ratio.notna() & (df.gap_ratio > 0)
a.bar(x[m], df.gap_ratio[m], color=BLUE, width=.6)
a.set_yscale('log', base=2); a.axhline(1, color='#c9d6ec', lw=1.4); a.set_xticks(x); a.set_xticklabels(lab, rotation=55, ha='right', fontsize=9)
a.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}×")); a.set_title('gap ratio: reproduced gap ÷ reported gap', loc='left', fontweight='bold')
flag = df.reliability.eq('below_host_floor') & m
for xi in x[flag]: a.bar(xi, df.gap_ratio[xi], color='none', edgecolor='#e5533f', linestyle='--', width=.6)

# 2. gaps per PR: reported vs reproduced, relative to baseline
a = ax[0, 1]; w = .38
a.bar(x - w/2, df.reported_gap.fillna(0) * 100, w, color=ORANGE, label='reported gap')
a.bar(x + w/2, df.reproduced_gap * 100, w, color=BLUE, label='reproduced gap')
a.axhline(0, color='#c9d6ec', lw=1.4); a.set_xticks(x); a.set_xticklabels(lab, rotation=55, ha='right', fontsize=9)
a.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.0f}%")); a.set_title('gaps: (baseline − PR\'s solution) ÷ baseline', loc='left', fontweight='bold'); a.legend(frameon=False, fontsize=8)

# per field
g = df.groupby('field')
pf = pd.DataFrame({'n': g.size(), 'gap_ratio_med': g.gap_ratio.median(), 'gap_ratio_min': g.gap_ratio.min(), 'gap_ratio_max': g.gap_ratio.max(),
                   'reported_gap_med': g.reported_gap.median(), 'reproduced_gap_med': g.reproduced_gap.median()}).sort_index()
fx = np.arange(len(pf))
a = ax[1, 0]; mm = pf.gap_ratio_med.notna() & (pf.gap_ratio_med > 0)
a.bar(fx[mm.values], pf.gap_ratio_med[mm], color=BLUE, width=.5)
a.errorbar(fx[mm.values], pf.gap_ratio_med[mm], yerr=[(pf.gap_ratio_med - pf.gap_ratio_min)[mm], (pf.gap_ratio_max - pf.gap_ratio_med)[mm]], fmt='none', ecolor='#172a5c', capsize=3)
a.set_yscale('log', base=2); a.axhline(1, color='#c9d6ec', lw=1.4); a.set_xticks(fx); a.set_xticklabels([f"{f} (n={n})" for f, n in zip(pf.index, pf.n)], rotation=35, ha='right', fontsize=9)
a.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}×")); a.set_title('gap ratio per field (median, whisker = min–max)', loc='left', fontweight='bold')
a = ax[1, 1]
a.bar(fx - w/2, pf.reported_gap_med.fillna(0) * 100, w, color=ORANGE, label='reported gap (median)')
a.bar(fx + w/2, pf.reproduced_gap_med * 100, w, color=BLUE, label='reproduced gap (median)')
a.axhline(0, color='#c9d6ec', lw=1.4); a.set_xticks(fx); a.set_xticklabels(pf.index, rotation=35, ha='right', fontsize=9)
a.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.0f}%")); a.set_title('gaps per field (median)', loc='left', fontweight='bold'); a.legend(frameon=False, fontsize=8)
for aa in ax.flat: aa.grid(axis='y', color='#eef2f9'); aa.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()
pf
